In [16]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [17]:
df = pd.read_csv("../data/feature_engineered.csv")

df.head()

,Order,PID,MS SubClass,MS Zoning,Lot Frontage,Lot Area,Street,Alley,Lot Shape,Land Contour,...,Sale Condition,SalePrice,House Age,Remodeled,Has Pool,Has Garage,Has Fireplace,Total Area,Total Bath,Quality_Area
0,1,526301100,20,RL,141.0,31770,Pave,NaN,IR1,Lvl,...,Normal,215000,66,0,0,1,1,2736.0,2.0,9936
1,2,526350040,20,RH,80.0,11622,Pave,NaN,Reg,Lvl,...,Normal,105000,65,0,0,1,0,1778.0,1.0,4480
2,3,526351010,20,RL,81.0,14267,Pave,NaN,IR1,Lvl,...,Normal,172000,68,0,0,1,0,2658.0,1.5,7974
3,4,526353030,20,RL,93.0,11160,Pave,NaN,Reg,Lvl,...,Normal,244000,58,0,0,1,1,4220.0,3.5,14770
4,5,527105010,60,RL,74.0,13830,Pave,NaN,IR1,Lvl,...,Normal,189900,29,1,0,1,1,2557.0,2.5,8145


In [18]:
y = df["SalePrice"]
X = df.drop("SalePrice", axis=1)

In [19]:
X = pd.get_dummies(X)

X = X.fillna(
    X.median(numeric_only=True)
)

In [20]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

In [21]:
#std

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [22]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(

    X_test_scaled,

    dtype=torch.float32

)

y_train_tensor = torch.tensor(

    y_train.values,

    dtype=torch.float32

).view(-1, 1)

y_test_tensor = torch.tensor(

    y_test.values,

    dtype=torch.float32

).view(-1, 1)

In [23]:
class HousePriceMLP(nn.Module):
    def __init__(self,input_dim):
        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(input_dim,128),
            nn.ReLU(),
            nn.Linear(128,64),
            nn.ReLU(),
            nn.Linear(64,32),
            nn.ReLU(),
            nn.Linear(32,1)
        )
    def forward(self,x):
        return self.net(x)



In [24]:
input_dim = X_train_tensor.shape[1]

model = HousePriceMLP(input_dim)
model

HousePriceMLP(
  (net): Sequential(
    (0): Linear(in_features=313, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
  )
)

In [25]:
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [30]:
train_losses = []
test_losses = []
epochs = 5000

for epoch in range(epochs):
    #train
    model.train()

    pred_train = model(X_train_tensor)
    train_loss = loss_fn(
        pred_train,
        y_train_tensor
    )

    optimizer.zero_grad()
    train_loss.backward()
    optimizer.step()

    #test
    model.eval()

    with torch.no_grad():

        pred_test = model(X_test_tensor)

        test_loss = loss_fn(
            pred_test,
            y_test_tensor

        )

    train_losses.append(train_loss.item())
    test_losses.append(test_loss.item())

    if (epoch + 1) % 100 == 0:

        print(
            f"Epoch {epoch+1}, "
            f"Train Loss:{train_loss.item():.0f}, "
            f"Test Loss:{test_loss.item():.0f}"
        )

Epoch 100, Train Loss:36750618624, Test Loss:42563653632
Epoch 200, Train Loss:14061873152, Test Loss:14375452672
Epoch 300, Train Loss:3397775360, Test Loss:3973338112
Epoch 400, Train Loss:1918971648, Test Loss:2758727168
Epoch 500, Train Loss:1425847808, Test Loss:2284844032
Epoch 600, Train Loss:1144747264, Test Loss:1994219136
Epoch 700, Train Loss:950998464, Test Loss:1793961216
Epoch 800, Train Loss:805922176, Test Loss:1645483520
Epoch 900, Train Loss:693192384, Test Loss:1530854144
Epoch 1000, Train Loss:602106432, Test Loss:1438188416
Epoch 1100, Train Loss:527299744, Test Loss:1359225088
Epoch 1200, Train Loss:465511488, Test Loss:1293187328
Epoch 1300, Train Loss:414143904, Test Loss:1238181888
Epoch 1400, Train Loss:371278176, Test Loss:1192813056
Epoch 1500, Train Loss:334857504, Test Loss:1155292544
Epoch 1600, Train Loss:303691552, Test Loss:1124181632
Epoch 1700, Train Loss:277029632, Test Loss:1099767552
Epoch 1800, Train Loss:254169328, Test Loss:1080904320
Epoch 190

In [31]:
model.eval()

with torch.no_grad():
    pred_mlp_tensor = model(X_test_tensor)

pred_mlp = pred_mlp_tensor.numpy().flatten()

rmse_mlp = np.sqrt(mean_squared_error(y_test, pred_mlp))
mae_mlp = mean_absolute_error(y_test, pred_mlp)
r2_mlp = r2_score(y_test, pred_mlp)

print("MLP RMSE:", rmse_mlp)
print("MLP MAE:", mae_mlp)
print("MLP R2:", r2_mlp)

MLP RMSE: 31812.83665440729
MLP MAE: 17616.9140625
MLP R2: 0.8737698197364807
